<a href="https://colab.research.google.com/github/mrunmayee3108/NeuroSolve/blob/main/data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import re
from datasets import load_dataset

In [20]:
def clean_target_value(raw_ans):
    ans = raw_ans.replace(',', '').strip()
    if '/' in ans:
        try:
            num, den = ans.split('/')
            return float(num) / float(den)
        except:
            return None
    try:
        return float(ans)
    except ValueError:
        return None

In [21]:
dataset_algebra = load_dataset("EleutherAI/hendrycks_math", "algebra", split="test", trust_remote_code=True)
dataset_inter_algebra = load_dataset("EleutherAI/hendrycks_math", "intermediate_algebra", split="test")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'EleutherAI/hendrycks_math' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'EleutherAI/hendrycks_math' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


In [22]:
from datasets import concatenate_datasets
dataset = concatenate_datasets([dataset_algebra, dataset_inter_algebra])
df = dataset.to_pandas()

In [23]:
df.shape

(2090, 4)

In [24]:
valid_rows = []
for _, row in df.iterrows():
    match = re.search(r'\\boxed\{(.*?)\}', str(row['solution']))
    if match:
        raw_ans = match.group(1)
        cleaned_ans = clean_target_value(raw_ans)
        if cleaned_ans is not None:
            valid_rows.append({"question": row['problem'], "answer": cleaned_ans})
    if len(valid_rows) == 250:
        break

In [25]:
df_nonlinear = pd.DataFrame(valid_rows)
df_nonlinear.to_csv('nonlinear_test.csv', index=False)